# UK Supermarket Data Cleaning

## Project Overview

This notebook performs data cleaning on the supermarket datasets based on the findings from the data profiling stage.

Only issues that were investigated and justified during the profiling notebook are cleaned.

The objective is to improve data quality while preserving valid information.

In [1]:
import pandas as pd

In [2]:
aldi = pd.read_csv("../data/raw/All_Data_Aldi.csv")

asda = pd.read_csv("../data/raw/All_Data_ASDA.csv")

morrisons = pd.read_csv("../data/raw/All_Data_Morrisons.csv")

sains = pd.read_csv("../data/raw/All_Data_Sains.csv")

tesco = pd.read_csv("../data/raw/All_Data_Tesco.csv")

C:\Users\Atq12\AppData\Local\Temp\ipykernel_22164\569542028.py:3: DtypeWarning: Columns (0: own_brand) have mixed types. Specify dtype option on import or set low_memory=False.
  asda = pd.read_csv("../data/raw/All_Data_ASDA.csv")


In [3]:
print("Aldi:", aldi.shape)

print("ASDA:", asda.shape)

print("Morrisons:", morrisons.shape)

print("Sainsbury's:", sains.shape)

print("Tesco:", tesco.shape)

Aldi: (464863, 8)
ASDA: (2456414, 8)
Morrisons: (1794065, 8)
Sainsbury's: (2600289, 8)
Tesco: (2213611, 8)


# Cleaning Log

Every cleaning action performed in this notebook will be documented.

| Issue | Action | Reason |
|-------|--------|--------|

# Cleaning Step 1: Remove Duplicate Rows (Tesco)

## Issue

The profiling notebook identified **23,828 duplicate rows** in the Tesco dataset.

Investigation confirmed that:

- The duplicate rows were identical across all columns.
- Every duplicated record appeared exactly twice.

Therefore, duplicate removal is justified.

In [4]:
print("Before cleaning:", tesco.shape)

Before cleaning: (2213611, 8)


In [5]:
tesco_clean = tesco.drop_duplicates()   

In [6]:
print("after cleaning:", tesco_clean.shape)

after cleaning: (2189783, 8)


In [7]:
tesco_clean.duplicated().sum()

np.int64(0)

## Result

The duplicate rows were successfully removed from the Tesco dataset.

Verification confirmed that:

- The cleaned dataset contains no duplicate rows.
- The original raw dataset remains unchanged.
- A cleaned version of the Tesco dataset has been created for subsequent analysis.

# Cleaning Log

| Issue | Action | Reason |
|-------|--------|--------|
| Tesco duplicate rows (23,828) | Removed duplicate rows using `drop_duplicates()` | Investigation confirmed the rows were exact duplicates. |

# Cleaning Step 2: Handle Missing Product Information (ASDA)

## Issue

During the data profiling stage, the ASDA dataset was found to contain **26 records with missing product names**.

Further investigation identified two groups:

- **7 records** with missing product name, selling price, unit price, unit and own-brand information.
- **19 records** where only the product name and own-brand fields were missing, while pricing information remained available.

The first group contains insufficient information for meaningful analysis and will be considered for removal.

The second group retains useful pricing information and will be preserved for further analysis.

In [8]:
asda_clean = asda.copy()

In [10]:
asda_clean[ 
    asda_clean["prices_(£)"].isnull() &
    asda_clean["names"].isnull()
]

,supermarket,prices_(£),prices_unit_(£),unit,names,date,category,own_brand
212170,ASDA,NaN,NaN,NaN,NaN,20240406,fresh_food,NaN
339302,ASDA,NaN,NaN,NaN,NaN,20240401,household,NaN
477644,ASDA,NaN,NaN,NaN,NaN,20240327,free-from,NaN
1338635,ASDA,NaN,NaN,NaN,NaN,20240222,free-from,NaN
1783046,ASDA,NaN,NaN,NaN,NaN,20240205,fresh_food,NaN
2331716,ASDA,NaN,NaN,NaN,NaN,20240113,fresh_food,NaN
2356760,ASDA,NaN,NaN,NaN,NaN,20240112,fresh_food,NaN


In [11]:
len(
    asda_clean[
        asda_clean["prices_(£)"].isnull() &
        asda_clean["names"].isnull()
    ]
)

7

In [13]:
asda_clean = asda_clean[
    ~(
        asda_clean["prices_(£)"].isnull() &
        asda_clean["names"].isnull()
    )   
]

In [15]:
len(
    asda_clean[
        asda_clean["prices_(£)"].isnull() & 
        asda_clean["names"].isnull()
    ]
)

0

In [16]:
print("ASDA after cleaning:", asda_clean.shape)

ASDA after cleaning: (2456407, 8)


## Result

The seven incomplete ASDA records were removed from the cleaned dataset.

These records contained insufficient information for analysis, with missing product name, selling price, unit price, unit and own-brand fields.

The remaining records with missing product names were retained because they still contain valid pricing information and may be useful for future analysis.

The original raw dataset remains unchanged.

| Issue | Action | Reason |
|-------|--------|--------|
| Tesco duplicate rows (23,828) | Removed duplicate rows using `drop_duplicates()` | Investigation confirmed the rows were exact duplicates. |
| ASDA incomplete records (7) | Removed incomplete records | Records contained insufficient information for meaningful analysis. |

# Cleaning Step 3: Retain Partially Complete ASDA Records

## Issue

Nineteen ASDA records contain missing product names and own-brand information but retain valid pricing, unit, date and category information.

## Decision

These records have been retained.

## Reason

Although the product names are unavailable, the remaining fields contain useful analytical information.

Removing these records would unnecessarily reduce the available pricing data.

The missing product names will remain as missing values and be documented as a limitation of the dataset.

# Cleaning Step 4: Aldi Missing Unit Information

## Issue

The Aldi dataset contains four records with missing unit price and unit information.

During the profiling stage, these records were found to belong to the same product.

This step investigates whether the missing values can be recovered from existing records.

In [21]:
aldi_clean = aldi.copy()

In [22]:
aldi_clean[aldi_clean["unit"].isnull()]

,supermarket,prices_(£),prices_unit_(£),unit,names,date,category,own_brand
337885,Aldi,0.65,NaN,NaN,Saxon Biodegradable Toilet Tissue Wipes - Fine...,20240205,household,False
342073,Aldi,0.65,NaN,NaN,Saxon Biodegradable Toilet Tissue Wipes - Fine...,20240204,household,False
455796,Aldi,0.65,NaN,NaN,Saxon Biodegradable Toilet Tissue Wipes - Fine...,20240110,household,False
461162,Aldi,0.65,NaN,NaN,Saxon Biodegradable Toilet Tissue Wipes - Fine...,20240109,household,False


In [24]:
aldi_clean[
    aldi_clean["names"].str.contains(
        "Saxon Biodegradable Toilet Tissue Wipes",
        na=False
    )
]

,supermarket,prices_(£),prices_unit_(£),unit,names,date,category,own_brand
337885,Aldi,0.65,NaN,NaN,Saxon Biodegradable Toilet Tissue Wipes - Fine...,20240205,household,False
342073,Aldi,0.65,NaN,NaN,Saxon Biodegradable Toilet Tissue Wipes - Fine...,20240204,household,False
455796,Aldi,0.65,NaN,NaN,Saxon Biodegradable Toilet Tissue Wipes - Fine...,20240110,household,False
461162,Aldi,0.65,NaN,NaN,Saxon Biodegradable Toilet Tissue Wipes - Fine...,20240109,household,False


## Decision

The four Aldi records with missing unit and unit price information were investigated.

No additional records for the same product were found with complete unit information.

As a result, the missing values cannot be recovered confidently from the available data.

The missing values have therefore been retained to avoid introducing unsupported or incorrect information into the dataset.

| Issue | Action | Reason |
|-------|--------|--------|
| Tesco duplicate rows (23,828) | Removed duplicate rows using `drop_duplicates()` | Investigation confirmed the rows were exact duplicates. |
| ASDA incomplete records (7) | Removed incomplete records | Records contained insufficient information for meaningful analysis. |
| ASDA partial records (19) | Retained records | Pricing information remained useful despite missing product names. |
| Aldi missing unit information (4) | Retained missing values | No reliable information was available to recover the missing values. |

# Cleaning Step 5: Morrisons Missing Unit Information

## Issue

The profiling stage identified **110 records** with missing unit and unit price information in the Morrisons dataset.

This step investigates whether these missing values can be recovered or whether they should be retained.

In [25]:
morrisons_clean = morrisons.copy()

In [26]:
morrisons_clean[morrisons_clean["unit"].isnull()]

,supermarket,prices_(£),prices_unit_(£),unit,names,date,category,own_brand
12782,Morrisons,8.5,NaN,NaN,Oyster Bay Pinot Noir,20240413,drinks,False
12790,Morrisons,11.0,NaN,NaN,Oyster Bay Pinot Grigio,20240413,drinks,False
12800,Morrisons,18.0,NaN,NaN,Penfolds Max Shiraz,20240413,drinks,False
12808,Morrisons,9.0,NaN,NaN,Maison Castel Bordeaux Rouge,20240413,drinks,False
24778,Morrisons,8.5,NaN,NaN,Oyster Bay Pinot Noir,20240412,drinks,False
...,...,...,...,...,...,...,...,...
520327,Morrisons,11.0,NaN,NaN,Oyster Bay Pinot Grigio 11.5%,20240315,drinks,False
546578,Morrisons,9.5,NaN,NaN,Oyster Bay Pinot Noir 11.5%,20240313,drinks,False
546580,Morrisons,18.0,NaN,NaN,Penfolds Max Shiraz,20240313,drinks,False
546595,Morrisons,7.5,NaN,NaN,Maison Castel Bordeaux Rouge,20240313,drinks,False


In [27]:
len(morrisons_clean[morrisons_clean["unit"].isnull()])

110

In [28]:
morrisons_clean[
    morrisons_clean["unit"].isnull()
]["names"].value_counts()

names
Maison Castel Bordeaux Rouge     28
Penfolds Max Shiraz              26
Oyster Bay Pinot Noir            23
Oyster Bay Pinot Grigio          23
Oyster Bay Pinot Noir 11.5%       5
Oyster Bay Pinot Grigio 11.5%     5
Name: count, dtype: int64

In [29]:
morrisons_clean[
    morrisons_clean["names"] == "Maison Castel Bordeaux Rouge"
]

,supermarket,prices_(£),prices_unit_(£),unit,names,date,category,own_brand
12808,Morrisons,9.0,NaN,NaN,Maison Castel Bordeaux Rouge,20240413,drinks,False
24801,Morrisons,9.0,NaN,NaN,Maison Castel Bordeaux Rouge,20240412,drinks,False
44154,Morrisons,9.0,NaN,NaN,Maison Castel Bordeaux Rouge,20240411,drinks,False
67339,Morrisons,9.0,NaN,NaN,Maison Castel Bordeaux Rouge,20240409,drinks,False
88153,Morrisons,9.0,NaN,NaN,Maison Castel Bordeaux Rouge,20240408,drinks,False
115120,Morrisons,9.0,NaN,NaN,Maison Castel Bordeaux Rouge,20240407,drinks,False
132289,Morrisons,9.0,NaN,NaN,Maison Castel Bordeaux Rouge,20240406,drinks,False
150926,Morrisons,9.0,NaN,NaN,Maison Castel Bordeaux Rouge,20240405,drinks,False
153432,Morrisons,9.0,NaN,NaN,Maison Castel Bordeaux Rouge,20240404,drinks,False
178343,Morrisons,9.0,NaN,NaN,Maison Castel Bordeaux Rouge,20240403,drinks,False


In [30]:
affected_products = morrisons_clean[
    morrisons_clean["unit"].isnull()
]["names"].unique()

morrisons_clean[
    morrisons_clean["names"].isin(affected_products)
].groupby("names")[["unit", "prices_unit_(£)"]].apply(
    lambda x: x.notnull().sum()
)

,unit,prices_unit_(£)
names,,
Maison Castel Bordeaux Rouge,0,0
Oyster Bay Pinot Grigio,0,0
Oyster Bay Pinot Grigio 11.5%,0,0
Oyster Bay Pinot Noir,0,0
Oyster Bay Pinot Noir 11.5%,0,0
Penfolds Max Shiraz,0,0


## Decision

The missing unit and unit price values in the Morrisons dataset were investigated.

The affected products were examined to determine whether complete records existed elsewhere in the dataset.

No complete unit or unit price information was found for any of the affected products.

Therefore, the missing values have been retained to preserve data integrity and avoid introducing unsupported values.

| Morrisons missing unit information (110) | Retained missing values | No reliable information was available to recover the missing values. |

# Cleaning Step 6: Sainsbury's Missing Unit Information

## Issue

The profiling stage identified **209 records** with missing unit and unit price information in the Sainsbury's dataset.

This step investigates whether the missing values can be recovered from other records or should be retained.

In [31]:
sains_clean = sains.copy()

In [32]:
len(sains_clean[sains_clean["unit"].isnull()])

209

In [33]:
sains_clean[
    sains_clean["unit"].isnull()
]["names"].value_counts()

names
Sainsbury's Hot Smoked Salmon & Baby Potato Salad, Taste the Difference     91
Sainsbury's Hot Cross Bun Loaf 400g                                         87
Sainsbury's Spicy Chicken & Puttanesca Pasta Salad, Taste the Difference    24
Cockburn's Fine White Port 75cl                                              7
Name: count, dtype: int64

In [34]:
affected_products = sains_clean[
    sains_clean["unit"].isnull()
]["names"].unique()

sains_clean[
    sains_clean["names"].isin(affected_products)
].groupby("names")[["unit", "prices_unit_(£)"]].apply(
    lambda x: x.notnull().sum()
)


,unit,prices_unit_(£)
names,,
Cockburn's Fine White Port 75cl,75,75
Sainsbury's Hot Cross Bun Loaf 400g,0,0
"Sainsbury's Hot Smoked Salmon & Baby Potato Salad, Taste the Difference",0,0
"Sainsbury's Spicy Chicken & Puttanesca Pasta Salad, Taste the Difference",67,67


In [35]:
sains_clean[
    sains_clean["names"] == "Cockburn's Fine White Port 75cl"
][["unit", "prices_unit_(£)"]].drop_duplicates()

,unit,prices_unit_(£)
13947,l,14.63
890968,l,18.62
2147603,NaN,NaN


In [36]:
sains_clean[
    sains_clean["names"] == "Sainsbury's Spicy Chicken & Puttanesca Pasta Salad, Taste the Difference"
][["unit", "prices_unit_(£)"]].drop_duplicates()

,unit,prices_unit_(£)
24704,kg,14.2
1936754,NaN,NaN


## Decision

Investigation of the Sainsbury's dataset showed that not all missing values required the same treatment.

- For **Cockburn's Fine White Port 75cl**, the unit was consistently recorded as `l`, but the price per unit varied across records. Therefore, only the missing unit values will be recovered.
- For **Sainsbury's Spicy Chicken & Puttanesca Pasta Salad**, both the unit (`kg`) and the unit price (`14.2`) were consistent across complete records. Therefore, both missing values will be recovered.
- For the remaining products, no complete records were available. Their missing values will be retained.

In [37]:
sains_clean.loc[
    (sains_clean["names"] == "Cockburn's Fine White Port 75cl") &
    (sains_clean["unit"].isnull()),
    "unit"
] = "l"

In [38]:
sains_clean.loc[
    (sains_clean["names"] == "Sainsbury's Spicy Chicken & Puttanesca Pasta Salad, Taste the Difference") &
    (sains_clean["unit"].isnull()),
    "unit"
] = "kg"

sains_clean.loc[
    (sains_clean["names"] == "Sainsbury's Spicy Chicken & Puttanesca Pasta Salad, Taste the Difference") &
    (sains_clean["prices_unit_(£)"].isnull()),
    "prices_unit_(£)"
] = 14.2

In [39]:
sains_clean[
    sains_clean["unit"].isnull()
]["names"].value_counts()

names
Sainsbury's Hot Smoked Salmon & Baby Potato Salad, Taste the Difference    91
Sainsbury's Hot Cross Bun Loaf 400g                                        87
Name: count, dtype: int64

## Result

The missing unit information in the Sainsbury's dataset was investigated on a product-by-product basis.

Two products contained sufficient complete records to recover missing values:

- **Cockburn's Fine White Port 75cl**: Missing unit values were filled with `l`. Unit prices were not filled because multiple valid values existed.
- **Sainsbury's Spicy Chicken & Puttanesca Pasta Salad, Taste the Difference**: Missing unit values were filled with `kg` and missing unit prices were filled with `14.2`, as these values were consistent across complete records.

For the remaining products, no reliable reference values were available. Their missing values were retained to preserve data integrity.

# Data Cleaning Summary

| Dataset | Changes Made |
|----------|--------------|
| Aldi | 0 rows removed, 0 values imputed |
| ASDA | 7 rows removed |
| Morrisons | 0 rows removed, 0 values imputed |
| Sainsbury's | 31 missing values recovered (24 pasta salad + 7 Cockburn's unit values) |
| Tesco | 23,828 duplicate rows removed |

## Overall Outcome

- Duplicate rows removed: 23,828
- Incomplete rows removed: 7
- Missing values recovered: 31
- Missing values intentionally retained where recovery was not supported by evidence.

In [40]:
import os

os.makedirs("../data/cleaned", exist_ok=True)

aldi_clean.to_csv("../data/cleaned/aldi_clean.csv", index=False)
asda_clean.to_csv("../data/cleaned/asda_clean.csv", index=False)
morrisons_clean.to_csv("../data/cleaned/morrisons_clean.csv", index=False)
sains_clean.to_csv("../data/cleaned/sains_clean.csv", index=False)
tesco_clean.to_csv("../data/cleaned/tesco_clean.csv", index=False)